# Linear Regression with Gradient Descent

**Course**: CMOR 438 / INDE 577 — Data Science & Machine Learning  
**Dataset**: International Football Results (1872–present)  
**Author**: Neriah29  

---

## What is Linear Regression?

The Perceptron predicted **categories** — win or not win. Linear Regression predicts **numbers** — a continuous value like goal difference, total goals scored, or temperature.

The core idea: given some input features, find the **straight line** (or flat plane, in higher dimensions) that best fits the data.

For football, a natural question is:
> *"Given what we know about two teams going in, how many more goals will the home team score than the away team?"*

That number — **goal difference** — is our target. It can be positive (home win), zero (draw), or negative (away win).

### The Model

Linear Regression makes predictions using exactly the same formula as the Perceptron:

$$\hat{y} = w_1 x_1 + w_2 x_2 + \ldots + w_n x_n + b$$

The critical difference: **we don't apply a step function at the end**. The raw number $\hat{y}$ *is* the prediction. That's what makes it a regression (continuous output) rather than a classification (discrete label).

---

## The Big New Idea: Gradient Descent

This is the most important concept in all of machine learning. Everything from here — Logistic Regression, Neural Networks, and beyond — is built on this.

### The problem we're solving

We have weights and a bias. We want to find the values that make our predictions as accurate as possible. But how do we *search* for those values? There are infinitely many possible combinations.

### Step 1: Define "how wrong" with a Loss Function

First, we need a single number that measures how bad our current predictions are. We use **Mean Squared Error (MSE)**:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

Take every prediction, subtract the real answer, square it (so sign doesn't matter and big errors are punished more), and average across all samples. **Lower MSE = better model.**

### Step 2: Imagine a landscape

Picture a landscape of hills and valleys. The horizontal position represents the current weight values. The height represents the MSE — how wrong you currently are.

You're standing somewhere in this landscape, blindfolded. Your goal: find the lowest valley (minimum MSE). You can't see the whole landscape — you can only feel the slope *right where you're standing*.

### Step 3: The gradient tells you which way is uphill

The **gradient** is the slope of the loss function at your current position. It's a vector of derivatives — one per weight — that points in the direction of **steepest ascent** (uphill).

This is where the calculus comes in. We take the derivative of MSE with respect to each weight:

$$\frac{\partial \text{MSE}}{\partial w_j} = \frac{-2}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i) \cdot x_{ij}$$

You don't need to memorize this derivation — what matters is what it *means*: for each weight, this tells us how much the error increases or decreases if we nudge that weight slightly.

### Step 4: Step in the opposite direction

Since the gradient points uphill, we step **downhill** — subtract it from the weights:

$$w \leftarrow w - \alpha \cdot \nabla_{w} \text{MSE}$$

Where $\alpha$ is the **learning rate** — how big of a step to take.

### Step 5: Repeat

Do this for many epochs. Each step lands you slightly lower in the valley. Eventually you reach the bottom — the minimum loss — and the weights stop changing meaningfully. That's convergence.

### Why not just jump straight to the bottom?

Because we don't know where it is. We can only see the local slope. Gradient descent is essentially feeling your way downhill in the dark, one small step at a time.

### How is this different from the Perceptron?

The Perceptron update rule was: *"were you wrong? nudge in the right direction by a fixed amount."* No calculus, no sense of degree of wrongness.

Gradient descent is: *"how wrong were you, and in exactly what direction does that wrongness pull each weight?"* It uses the actual shape of the error surface, which is why it's far more powerful and generalizable.


---
## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import sys
sys.path.insert(0, '../../src')
from football_ml.supervised_learning.linear_regression import LinearRegression

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
SEED = 42

---
## 1. Load & Explore the Dataset

In [ ]:
df = pd.read_csv('../../data/results.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
print(f'Shape: {df.shape}')
df.head()

---
## 2. Feature Engineering

### Target variable

For regression, we predict **goal difference**: home score minus away score.
- Positive → home team scored more
- Zero → draw
- Negative → away team scored more

This is a natural continuous output — perfect for Linear Regression.

### Features

Same engineered features as the Perceptron notebook, built with a rolling 10-game window to capture recent form.

In [ ]:
df['goal_diff'] = df['home_score'] - df['away_score']

def compute_team_rolling_stats(df, window=10):
    home_log = df[['date', 'home_team', 'home_score', 'away_score']].copy()
    home_log.columns = ['date', 'team', 'scored', 'conceded']
    away_log = df[['date', 'away_team', 'away_score', 'home_score']].copy()
    away_log.columns = ['date', 'team', 'scored', 'conceded']
    team_log = pd.concat([home_log, away_log]).sort_values('date').reset_index(drop=True)
    team_log['rolling_scored'] = (
        team_log.groupby('team')['scored']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    team_log['rolling_conceded'] = (
        team_log.groupby('team')['conceded']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    return team_log.drop_duplicates(subset=['date', 'team'], keep='last').set_index(['date', 'team'])

team_stats = compute_team_rolling_stats(df)

def get_stat(row, team_col, stat_col):
    try:
        return team_stats.loc[(row['date'], row[team_col]), stat_col]
    except KeyError:
        return np.nan

df['home_goals_rolling']    = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_scored'), axis=1)
df['home_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_conceded'), axis=1)
df['away_goals_rolling']    = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_scored'), axis=1)
df['away_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_conceded'), axis=1)

home_wins = df.groupby('home_team').apply(lambda g: (g['home_score'] > g['away_score']).mean()).rename('home_win_rate')
away_wins = df.groupby('away_team').apply(lambda g: (g['away_score'] > g['home_score']).mean()).rename('away_win_rate')
df = df.join(home_wins, on='home_team').join(away_wins, on='away_team')
df['neutral'] = df['neutral'].astype(int)

feature_cols = [
    'home_goals_rolling', 'away_goals_rolling',
    'home_conceded_rolling', 'away_conceded_rolling',
    'home_win_rate', 'away_win_rate',
    'neutral'
]
df_clean = df[feature_cols + ['goal_diff']].dropna()
print(f'Clean dataset: {df_clean.shape[0]} rows')
df_clean['goal_diff'].describe()

### Target Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df_clean['goal_diff'], bins=30, color='#4878CF', edgecolor='white', linewidth=0.8)
ax.axvline(0, color='black', linestyle='--', linewidth=1.2, label='Draw (0)')
ax.axvline(df_clean['goal_diff'].mean(), color='#E87272', linestyle='--',
           linewidth=1.2, label=f'Mean ({df_clean["goal_diff"].mean():.2f})')
ax.set_xlabel('Goal Difference (Home − Away)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Distribution of Goal Difference', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

The distribution is roughly bell-shaped and centered slightly above zero — home teams score more on average. This is the well-known **home advantage** effect in football.

---
## 3. Prepare Data

In [ ]:
X = df_clean[feature_cols].values
y = df_clean['goal_diff'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape} | Test: {X_test_sc.shape}')
print(f'Target range: [{y.min():.0f}, {y.max():.0f}] | Mean: {y.mean():.3f}')

---
## 4. Train the Model

In [ ]:
model = LinearRegression(learning_rate=0.05, n_epochs=1000, random_state=SEED)
model.fit(X_train_sc, y_train)

train_rmse = model.rmse(X_train_sc, y_train)
test_rmse  = model.rmse(X_test_sc,  y_test)
train_r2   = model.score(X_train_sc, y_train)
test_r2    = model.score(X_test_sc,  y_test)

print(f'Train RMSE: {train_rmse:.4f} goals | Test RMSE: {test_rmse:.4f} goals')
print(f'Train R²:   {train_r2:.4f}       | Test R²:   {test_r2:.4f}')

**RMSE** is in the same units as the target (goals), so it's directly interpretable: "on average, the model's goal difference prediction is off by X goals."

**R²** tells us how much of the variance in goal difference our features explain. An R² of 0.10 means the model explains 10% of the variation — low, but not surprising for football, which has enormous randomness.

---
## 5. Loss Curve — Watching Gradient Descent Work

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Full loss curve
axes[0].plot(model.loss_history_, color='#4878CF', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('MSE', fontsize=12)
axes[0].set_title('Loss Curve — All Epochs', fontsize=13, fontweight='bold')

# First 100 epochs zoomed in — the action happens fast
axes[1].plot(model.loss_history_[:100], color='#E87272', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('MSE', fontsize=12)
axes[1].set_title('Loss Curve — First 100 Epochs (zoomed)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

### What you're seeing

The left plot shows the full training run. The loss (MSE) falls quickly at first, then levels off. That leveling-off is **convergence** — the model has found (approximately) the bottom of the valley and additional epochs aren't helping.

The right plot zooms into the first 100 epochs. Notice how steep the drop is early on — the gradient is large when you're far from the minimum, so each step moves you a lot. As you approach the minimum, the gradient flattens and steps get smaller.

---
## 6. Visualize — Predicted vs Actual

In [ ]:
y_pred = model.predict(X_test_sc)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: predicted vs actual
axes[0].scatter(y_test, y_pred, alpha=0.3, s=10, color='#4878CF')
lim = max(abs(y_test).max(), abs(y_pred).max()) + 0.5
axes[0].plot([-lim, lim], [-lim, lim], 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Goal Difference', fontsize=12)
axes[0].set_ylabel('Predicted Goal Difference', fontsize=12)
axes[0].set_title('Predicted vs Actual', fontsize=13, fontweight='bold')
axes[0].legend()

# Residuals: prediction error distribution
residuals = y_test - y_pred
axes[1].hist(residuals, bins=40, color='#E87272', edgecolor='white', linewidth=0.5)
axes[1].axvline(0, color='black', linestyle='--', linewidth=1.2)
axes[1].set_xlabel('Residual (Actual − Predicted)', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Residual Distribution', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

### Reading these plots

**Left — Predicted vs Actual**: A perfect model would have all points on the red dashed line. The wider the scatter around that line, the worse the model. Notice how the model's predictions cluster near zero even when the actual values are extreme — it's hedging toward the average.

**Right — Residuals**: Residuals are the errors (actual minus predicted). A well-calibrated model has residuals centered at zero with a roughly symmetric bell shape. Systematic skew would indicate bias.

---
## 7. Feature Weights

In [ ]:
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#4878CF' if w > 0 else '#E87272' for w in model.weights_]
ax.barh(feature_cols, model.weights_, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Weight (scaled)', fontsize=12)
ax.set_title('Learned Feature Weights', fontsize=13, fontweight='bold')

pos_patch = mpatches.Patch(color='#4878CF', label='Increases predicted goal diff')
neg_patch = mpatches.Patch(color='#E87272', label='Decreases predicted goal diff')
ax.legend(handles=[pos_patch, neg_patch], fontsize=9)
plt.tight_layout()
plt.show()

---
## 8. Effect of Learning Rate

What happens if the learning rate is too large or too small?

In [ ]:
learning_rates = [0.001, 0.01, 0.05, 0.3]
colors_lr = ['#2ecc71', '#4878CF', '#e67e22', '#E87272']

fig, ax = plt.subplots(figsize=(9, 5))
for lr, color in zip(learning_rates, colors_lr):
    m = LinearRegression(learning_rate=lr, n_epochs=200, random_state=SEED)
    m.fit(X_train_sc, y_train)
    loss = m.loss_history_
    # Clip for display if diverged
    loss_clipped = np.clip(loss, 0, loss[0] * 3)
    ax.plot(loss_clipped, label=f'lr={lr}', color=color, linewidth=2)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('MSE (clipped)', fontsize=12)
ax.set_title('Effect of Learning Rate on Convergence', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### What to look for

- **Too small** (lr=0.001): converges very slowly — you're taking tiny steps and need many more epochs.
- **Just right** (lr=0.05): drops quickly and stabilizes.
- **Too large** (lr=0.3): can overshoot the minimum and oscillate, or even diverge (loss increases instead of decreasing).

Choosing a good learning rate is one of the most important practical decisions in training any ML model.

---
## 9. Discussion — Does Linear Regression Suit Football?

### What it does well
- **Interpretable**: each weight directly tells you how much a feature shifts the predicted goal difference.
- **Fast**: gradient descent on this size dataset runs in milliseconds.
- **Principled**: the loss function (MSE) has a clean mathematical meaning and gradient descent minimizes it systematically.
- **Quantitative output**: predicting goal difference is more informative than just win/lose.

### What it struggles with

1. **Football is highly non-linear.** The relationship between team form and goals isn't a straight line — it's messy, context-dependent, and full of surprises.

2. **Assumes additive, independent feature effects.** The model treats each feature separately. It can't capture compound effects like "home team attacks well *combined with* away team defending poorly."

3. **Sensitive to outliers.** MSE squares the errors, so a 7-0 result pulls the weights much harder than a 1-0 result. Football has plenty of outliers.

4. **Predicts a continuous number, not a class.** To use this for match prediction we'd need to threshold the output (>0 → home win, etc.), which is lossy.

### What comes next

Logistic Regression takes this exact same framework — weighted sum, gradient descent, loss function — but swaps MSE for a loss function designed for probability prediction. It directly outputs the *probability* of a win rather than a goal count. That's a more honest answer to the classification question.

---
## Summary

| | |
|---|---|
| **Algorithm type** | Regression (continuous output) |
| **Target variable** | Goal difference (home − away) |
| **Activation** | None — raw weighted sum is the output |
| **Loss function** | Mean Squared Error (MSE) |
| **Optimization** | Batch Gradient Descent |
| **Key metrics** | RMSE (interpretable), R² (explained variance) |
| **Football suitability** | Moderate — captures trends, misses non-linearity |
| **Key concept introduced** | Gradient Descent — the engine of modern ML |
